Imlo coursework

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
# code to use my gpu
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

cuda


In [3]:
#transform to the train dataset with augmentation
train_transform = transforms.Compose([
    # adding random augmentations
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# transforms for the val dataset without data augmentations
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [4]:
#defining the train dataset
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = train_transform,
    download = True
)

#defining the val dataset
val_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = val_transform,
    download = False
)

#split will be 80:20
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size


train_indices, val_indices = torch.utils.data.random_split(
    range(len(train_data)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

train_data = torch.utils.data.Subset(
    train_data,
    train_indices.indices
)

val_data = torch.utils.data.Subset(
    val_data,
    val_indices.indices
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)

100%|██████████| 792M/792M [00:34<00:00, 22.7MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 8.78MB/s]


In [14]:
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])


# defining the test dataset
test_data = torchvision.datasets.OxfordIIITPet(
    root='./data',
    split='test',
    transform=test_transform,  # no random augmentations
    download=True
)

# dataloader for test data
test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    num_workers=2
)


In [5]:
image, label = train_data[0]

In [6]:
image.size()

torch.Size([3, 128, 128])

In [7]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [8]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)


        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)


        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)


        self.pool = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64 * 32 * 32, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 37)

    def forward(self, input):
        input = F.relu(self.bn1(self.conv1(input))) #conv1 then relu
        input = self.pool(input) #1st max pool

        input = F.relu(self.bn2(self.conv2(input))) #conv2 then relu

        input = F.relu(self.bn3(self.conv3(input))) #conv3 then relu
        input = F.relu(self.bn4(self.conv4(input))) #conv4 then relu
        input = self.pool(input) #2nd max pool

        input = torch.flatten(input, 1)  #flattening
        input = F.relu(self.fc1(input))  #applying fc1, then RELU
        input = F.relu(self.fc2(input))  #applying fc2, then RELU
        input = self.dropout(input)  #applying dropout
        input = self.fc3(input)  #applying fc3
        return input

In [9]:
# defining the NN itself
network = NeuralNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.0001, weight_decay=0.0001)

In [10]:
# training the model
for epoch in range(30):
    print("Training epoch:", epoch)
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = network(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

        #training accuracy
        predicted = outputs.argmax(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


    running_loss_calc = running_loss / len(train_loader)
    training_accuracy = 100 * correct / total

    print("Loss:", running_loss_calc)
    print("Training accuracy:", training_accuracy)

Training epoch: 0
Loss: 3.6703931818837705
Training accuracy: 2.989130434782609
Training epoch: 1
Loss: 3.5682868646538775
Training accuracy: 4.6875
Training epoch: 2
Loss: 3.5296712958294414
Training accuracy: 5.944293478260869
Training epoch: 3
Loss: 3.4759166914483774
Training accuracy: 6.6915760869565215
Training epoch: 4
Loss: 3.417466288027556
Training accuracy: 7.846467391304348
Training epoch: 5
Loss: 3.4075847967811255
Training accuracy: 8.695652173913043
Training epoch: 6
Loss: 3.3448502239973648
Training accuracy: 9.918478260869565
Training epoch: 7
Loss: 3.3130198665287183
Training accuracy: 9.918478260869565
Training epoch: 8
Loss: 3.2552311420440674
Training accuracy: 10.971467391304348
Training epoch: 9
Loss: 3.213659317597099
Training accuracy: 12.703804347826088
Training epoch: 10
Loss: 3.17148726919423
Training accuracy: 13.451086956521738
Training epoch: 11
Loss: 3.1602166217306387
Training accuracy: 13.179347826086957
Training epoch: 12
Loss: 3.0954735175423
Trainin

In [12]:
torch.save(network.state_dict(), "model.pth")

In [13]:
!ls

data  model.pth  sample_data


In [15]:
# testing the model on test data
correct = 0
total = 0

network.eval()

with torch.no_grad():
  for images, labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)


    outputs = network(images)
    predicted = outputs.argmax(1)

    total += len(labels)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 20.63232488416462
